In [83]:
import pandas as pd

In [84]:
# 1️⃣ Cargar el archivo CSV original
df = pd.read_csv("Mexico.csv", encoding="utf-8") 

columnas_seleccionadas = [
    "id", "host_name", "host_since", "host_location", "host_response_time",
    "host_response_rate", "host_acceptance_rate", "host_is_superhost",
    "host_neighbourhood", "host_total_listings_count", "host_verifications",
    "host_has_profile_pic", "host_identity_verified", "neighbourhood",
    "neighbourhood_cleansed", "latitude", "longitude", "property_type",
    "room_type", "accommodates", "bathrooms", "bathrooms_text", "bedrooms",
    "beds", "amenities", "price", "minimum_nights", "maximum_nights",
    "minimum_nights_avg_ntm", "maximum_nights_avg_ntm", "has_availability",
    "availability_30", "availability_60", "availability_90",
    "availability_365", "number_of_reviews", "number_of_reviews_ltm",
    "number_of_reviews_l30d", "first_review", "last_review",
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication", "review_scores_location",
    "review_scores_value", "license", "instant_bookable", "reviews_per_month"
]

df = df[columnas_seleccionadas]

df["host_location"] = df["host_location"].fillna("No location")
df["host_response_time"] = df["host_response_time"].fillna("Not defined")
df["host_response_rate"] = df["host_response_rate"].fillna("0%")#
df["host_acceptance_rate"] = df["host_acceptance_rate"].fillna("0%")#
df["host_neighbourhood"] = df["host_neighbourhood"].fillna("Not specified")
df["neighbourhood"] = df["neighbourhood"].fillna("No specified")
df["first_review"] = df["first_review"].fillna("31/12/2030")
df["last_review"] = df["last_review"].fillna("31/12/2030")
df["license"] = df["license"].fillna("Without license")
df["bathrooms_text"] = df["bathrooms_text"].fillna("0 zero")#
df["host_identity_verified"] = df["host_identity_verified"].fillna("f")#
df["has_availability"] = df["has_availability"].fillna("f")#
df["instant_bookable"] = df["instant_bookable"].fillna("f")#
df["host_is_superhost"] = df["host_is_superhost"].fillna("f")#

df['price'] = df['price'].str.lstrip('$').str.replace(',', '').astype(float)

# 2️⃣ Función para reemplazar "Desconocido" con media, mediana o moda
def reemplazar_desconocido(col):
    if col.dtype in ['int64', 'float64']:  # Si es numérica
        if col.isnull().sum() < len(col) / 2:  # Si hay suficientes datos válidos
            return col.replace("Desconocido", col.median())  # Usa la mediana
        else:
            return col.replace("Desconocido", col.mean())  # Usa la media
    else:  # Si es categórica
        return col.replace("Desconocido", col.mode()[0])  # Usa la moda

# 3️⃣ Aplicar la función a todas las columnas
df = df.apply(reemplazar_desconocido)

# 4️⃣ Eliminar comas "," en todas las filas
df = df.replace(",", "", regex=True)
df = df.replace("%", "", regex=True)
df = df.replace("$", "", regex=True)

# 8️⃣ Columnas donde aún hay "Desconocido"
columnas_a_corregir = ['price', 'first_review', 'last_review']

# 9️⃣ Aplicar la función solo a las columnas específicas
df[columnas_a_corregir] = df[columnas_a_corregir].apply(reemplazar_desconocido)

# 🔟 Guardar la versión final

print("Paso 2 completado: Se han reemplazado valores 'Desconocido' en las columnas específicas.")

# 🔟 Verificar si aún existen "Desconocido" o ","
columnas_desconocido = []
columnas_coma = []

for columna in df.columns:
    if df[columna].astype(str).str.contains("Desconocido", na=False).any():
        columnas_desconocido.append(columna)
    if df[columna].astype(str).str.contains(",", na=False).any():
        columnas_coma.append(columna)

if columnas_desconocido:
    print("❌ Columnas que aún contienen 'Desconocido':", columnas_desconocido)
else:
    print("✅ No se encontraron valores 'Desconocido' en ninguna columna.")

if columnas_coma:
    print("❌ Columnas que aún contienen ',':", columnas_coma)
else:
    print("✅ No se encontraron comas ',' en ninguna columna.")

print("✔ Archivo final limpio guardado como RepChecaLimp3.csv")
cuantitativas = df.iloc[ : , [0, 9, 15, 16, 19, 20, 22, 23, 26, 27, 28, 29, 31, 32, 33, 34, 35, 36, 37, 40, 41, 42, 43, 44, 45, 46, 49]]
cualitativas = df.iloc[ : , [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 17, 18, 21, 24, 25, 30, 38, 39, 47, 48]]
y = cuantitativas
Limite_Superior = y.mean() + 3*y.std()
Limite_Inferior = y.mean() - 3*y.std()
data2 = cuantitativas[(y <= Limite_Superior) & (y >= Limite_Inferior)]
data_clean = data2.copy()
data_clean = data_clean.fillna(round(data2.mean(), 1))
y = cuantitativas
percentile25 = y.quantile(0.25) #Q1
percentile75 = y.quantile(0.75) #Q3
iqr = percentile75 - percentile25
Limite_Superior_iqr = percentile75 + 1.5*iqr
Limite_Inferior_iqr = percentile25 - 1.5*iqr
data2_iqr = cuantitativas[(y <= Limite_Superior_iqr) & (y>= Limite_Inferior_iqr)]
df_iqr = data2_iqr.copy()
df_iqr = df_iqr.fillna(round(data2_iqr.mean(), 1))
Datos_limpios = pd.concat([cualitativas, df_iqr], axis = 1)
df = Datos_limpios
df.to_csv("Mexico.csv", index=0)

Paso 2 completado: Se han reemplazado valores 'Desconocido' en las columnas específicas.
✅ No se encontraron valores 'Desconocido' en ninguna columna.
✅ No se encontraron comas ',' en ninguna columna.
✔ Archivo final limpio guardado como RepChecaLimp3.csv


In [85]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26281 entries, 0 to 26280
Data columns (total 50 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   host_name                    26278 non-null  object 
 1   host_since                   26278 non-null  object 
 2   host_location                26281 non-null  object 
 3   host_response_time           26281 non-null  object 
 4   host_response_rate           26281 non-null  int64  
 5   host_acceptance_rate         26281 non-null  int64  
 6   host_is_superhost            26281 non-null  object 
 7   host_neighbourhood           26281 non-null  object 
 8   host_verifications           26278 non-null  object 
 9   host_has_profile_pic         26278 non-null  object 
 10  host_identity_verified       26281 non-null  object 
 11  neighbourhood                26281 non-null  object 
 12  neighbourhood_cleansed       26281 non-null  object 
 13  property_type   